In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [2]:
i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.

In [3]:
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\potato"

In [4]:
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    

Found 2000 files belonging to 2 classes.
Using 1600 files for training.


In [5]:
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set

Found 2000 files belonging to 2 classes.
Using 400 files for validation.


In [6]:
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)

In [7]:
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())

Classes: ['Potato___Early_Blight', 'Potato___Late_Blight']
Train batches: 50
Val batches: 7
Test batches: 6


In [8]:
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)

In [9]:
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [10]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers


In [11]:
num = len(class_names)


In [12]:
base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False

In [13]:
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint

In [14]:
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

In [15]:
x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)

In [16]:
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)

In [17]:
model = models.Model(inputs, outputs)

In [18]:
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]

In [20]:
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 941ms/step - accuracy: 0.7485 - loss: 0.5793

50/50 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.8581 - loss: 0.3396 - val_accuracy: 0.9712 - val_loss: 0.2173 - learning_rate: 0.0010
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 902ms/step - accuracy: 0.9625 - loss: 0.0894

50/50 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.9688 - loss: 0.0768 - val_accuracy: 0.9808 - val_loss: 0.1382 - learning_rate: 0.0010
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 893ms/step - accuracy: 0.9629 - loss: 0.0811

50/50 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.9700 - loss: 0.0679 - val_accuracy: 0.9856 - val_loss: 0.0945 - learning_rate: 0.0010
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 911ms/step - accuracy: 0.9857 - loss: 0.0462

50/50 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.9819 - loss: 0.0488 - val_accuracy: 0.9856 - val_loss: 0.0510 - learning_rate: 0.0010
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 943ms/step - accuracy: 0.9690 - loss: 0.0705

50/50 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.9775 - loss: 0.0555 - val_accuracy: 0.9856 - val_loss: 0.0446 - learning_rate: 0.0010
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 981ms/step - accuracy: 0.9835 - loss: 0.0511

50/50 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.9875 - loss: 0.0387 - val_accuracy: 0.9952 - val_loss: 0.0253 - learning_rate: 0.0010
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.9850 - loss: 0.0443 - val_accuracy: 0.9904 - val_loss: 0.0315 - learning_rate: 0.0010
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9877 - loss: 0.0341

50/50 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.9875 - loss: 0.0333 - val_accuracy: 0.9904 - val_loss: 0.0239 - learning_rate: 0.0010
Epoch 9/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.9781 - loss: 0.0535 - val_accuracy: 0.9808 - val_loss: 0.0315 - learning_rate: 0.0010
Epoch 10/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9889 - loss: 0.0273

50/50 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.9869 - loss: 0.0279 - val_accuracy: 0.9904 - val_loss: 0.0204 - learning_rate: 0.0010


In [21]:
model.save("potato.keras")

In [22]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 866ms/step - accuracy: 0.9948 - loss: 0.0107
Test Loss: 0.0107
Test Accuracy: 99.48%
